<a href="https://colab.research.google.com/github/shalini2004singh/ecommerce-delivery-analysis/blob/main/notebooks/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!find /content/drive/MyDrive -iname "*olist*"

In [9]:
import pandas as pd

orders = pd.read_csv("/content/olist_orders_dataset.csv")
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors="coerce")

print(orders.shape)
print(orders.isna().sum())
print(orders["order_id"].duplicated().sum())
print(orders["order_status"].value_counts())

(99441, 8)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
0
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [10]:
delivered = orders[orders["order_status"] == "delivered"]

print("Delivered but missing delivery date:", delivered["order_delivered_customer_date"].isna().sum())
print("Delivered before purchase:", (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print("Carrier date before purchase:", (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum())
print("Delivered to customer before carrier:", (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum())
print("Purchase date range:", orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())

days = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
print(days.describe())

Delivered but missing delivery date: 8
Delivered before purchase: 0
Carrier date before purchase: 166
Delivered to customer before carrier: 23
Purchase date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
dtype: float64


In [11]:
import os
os.makedirs("/content/interim", exist_ok=True)

orders["flag_missing_delivery_date"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()
orders["flag_bad_carrier_ts"] = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
orders["flag_delivered_before_carrier"] = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

orders.to_csv("/content/interim/orders_flagged.csv", index=False)
print(orders[["flag_missing_delivery_date", "flag_bad_carrier_ts", "flag_delivered_before_carrier"]].sum())

flag_missing_delivery_date         8
flag_bad_carrier_ts              166
flag_delivered_before_carrier     23
dtype: int64
